# 🚀 Python Performance Optimization: High-Performance Code

Welcome to **Python Mastery**! This notebook covers **performance optimization** - techniques and tools to make your Python code run faster, use less memory, and scale better.

## 📋 What You'll Learn
- Profiling and benchmarking Python code
- Memory optimization and leak detection
- Algorithm optimization and Big O analysis
- Using efficient data structures and libraries
- Multiprocessing and parallel processing
- Caching and memoization techniques
- JIT compilation and optimization tools
- Database and I/O optimization

## 🎯 Learning Objectives
By the end of this notebook, you'll be able to:
- Profile Python applications to identify bottlenecks
- Optimize memory usage and detect memory leaks
- Choose optimal data structures and algorithms
- Implement caching and memoization strategies
- Use multiprocessing for CPU-bound tasks
- Optimize database queries and I/O operations
- Apply JIT compilation and performance tools
- Balance performance optimization with code maintainability

---

## 📊 Profiling and Benchmarking

Before optimizing, you need to measure performance to identify bottlenecks.

In [ ]:
# Basic timing and benchmarking
print("=== BASIC TIMING AND BENCHMARKING ===")

import time
import timeit

# Simple timing example
def slow_function(n):
    """A deliberately slow function for demonstration."""
    result = 0
    for i in range(n):
        result += i ** 2
    return result

# Time the function
start_time = time.time()
result = slow_function(100000)
end_time = time.time()

print(f"Result: {result}")
print(f"Execution time: {end_time - start_time:.4f} seconds")

# Using timeit for more accurate measurements
print("\n=== USING TIMEIT FOR ACCURATE BENCHMARKING ===")

# Timeit automatically handles timing precision and runs multiple times
setup_code = """\
def slow_function(n):
    result = 0
    for i in range(n):
        result += i ** 2
    return result
"""

test_code = "slow_function(10000)"

# Run the benchmark
times = timeit.repeat(test_code, setup=setup_code, number=10, repeat=3)
print(f"Best time: {min(times):.4f} seconds")
print(f"Average time: {sum(times)/len(times):.4f} seconds")
print(f"All times: {[f'{t:.4f}s' for t in times]}")

# Comparing different implementations
print("\n=== COMPARING DIFFERENT IMPLEMENTATIONS ===")

def implementation1(n):
    """Using a loop."""
    result = 0
    for i in range(n):
        result += i ** 2
    return result

def implementation2(n):
    """Using list comprehension."""
    return sum([i ** 2 for i in range(n)])

def implementation3(n):
    """Using generator expression."""
    return sum(i ** 2 for i in range(n))

def implementation4(n):
    """Using built-in sum with range."""
    return sum(i*i for i in range(n))

def implementation5(n):
    """Using mathematical formula."""
    return n * (n - 1) * (2 * n - 1) // 6

# Benchmark all implementations
implementations = {
    'Loop': implementation1,
    'List Comp': implementation2,
    'Generator': implementation3,
    'Built-in': implementation4,
    'Formula': implementation5
}

n = 10000
results = {}

for name, func in implementations.items():
    # Verify correctness first
    result = func(n)
    if name == 'Loop':
        correct_result = result
    elif result != correct_result:
        print(f"{name}: INCORRECT RESULT ({result} != {correct_result})")
        continue
    
    # Benchmark
    time_taken = timeit.timeit(lambda: func(n), number=100)
    results[name] = time_taken
    print(f"{name}: {time_taken:.6f}s")

# Find the fastest
fastest = min(results, key=results.get)
print(f"\nFastest implementation: {fastest} ({results[fastest]:.6f}s)")
print(f"Formula is {results['Loop'] / results['Formula']:.1f}x faster than loop!")

In [ ]:
# Using cProfile for detailed profiling
print("\n=== USING CPROFILE FOR DETAILED PROFILING ===")

import cProfile
import pstats
from io import StringIO

# Function to profile
def complex_function():
    """A complex function with multiple operations."""
    data = []
    
    # Simulate some work
    for i in range(1000):
        data.append(i * i)
        data.sort()  # This is expensive!
    
    # More operations
    result = sum(data)
    filtered = [x for x in data if x % 2 == 0]
    
    return result, len(filtered)

# Profile the function
profiler = cProfile.Profile()
profiler.enable()

result = complex_function()

profiler.disable()

# Create a statistics object
stats = pstats.Stats(profiler)
stats.sort_stats('cumulative')

# Print the top 10 functions
print("Top 10 most time-consuming functions:")
stats.print_stats(10)

print(f"\nFunction result: {result}")

In [ ]:
# Memory profiling
print("\n=== MEMORY PROFILING ===")

# Check if memory_profiler is available
try:
    from memory_profiler import profile, memory_usage
    MEMORY_PROFILER_AVAILABLE = True
except ImportError:
    MEMORY_PROFILER_AVAILABLE = False
    print("memory_profiler not available - installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'memory_profiler'])
    from memory_profiler import profile, memory_usage
    MEMORY_PROFILER_AVAILABLE = True

if MEMORY_PROFILER_AVAILABLE:
    @profile
    def memory_intensive_function():
        """A function that uses a lot of memory."""
        big_list = []
        
        for i in range(100000):
            big_list.append(i)
            
        # Create a large string
        big_string = "x" * 1000000
        
        # Some processing
        result = sum(big_list)
        
        return result, len(big_string)
    
    print("Running memory-intensive function (with profiling):")
    result = memory_intensive_function()
    print(f"Result: {result}")
else:
    print("Memory profiler not available for demonstration")

## 🔧 Algorithm Optimization and Big O Analysis

Understanding algorithmic complexity helps you choose the right approach for performance-critical code.

In [ ]:
# Big O complexity examples
print("=== BIG O COMPLEXITY ANALYSIS ===")

import timeit

# O(1) - Constant time
def get_first_element(arr):
    """O(1) - constant time access."""
    return arr[0] if arr else None

# O(n) - Linear time
def find_max_loop(arr):
    """O(n) - linear search."""
    if not arr:
        return None
    max_val = arr[0]
    for num in arr:
        if num > max_val:
            max_val = num
    return max_val

# O(n²) - Quadratic time (inefficient)
def bubble_sort(arr):
    """O(n²) - bubble sort algorithm."""
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr

# O(log n) - Logarithmic time
def binary_search(arr, target):
    """O(log n) - binary search algorithm."""
    left, right = 0, len(arr) - 1
    
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    
    return -1

# Benchmark different complexities
sizes = [100, 1000, 10000]

print("Performance comparison for different input sizes:")
print("Size\t\tO(1)\t\tO(n)\t\tO(n²)\t\tO(log n)")
print("-" * 70)

for size in sizes:
    # Prepare test data
    test_list = list(range(size))
    test_list_sorted = sorted(test_list)
    
    # O(1)
    time_o1 = timeit.timeit(lambda: get_first_element(test_list), number=1000)
    
    # O(n)
    time_on = timeit.timeit(lambda: find_max_loop(test_list), number=100)
    
    # O(n²) - only for small sizes to avoid long wait
    if size <= 1000:
        test_small = test_list[:100]  # Use smaller list for O(n²)
        time_on2 = timeit.timeit(lambda: bubble_sort(test_small.copy()), number=10)
    else:
        time_on2 = float('inf')  # Too slow for large sizes
    
    # O(log n)
    time_ologn = timeit.timeit(lambda: binary_search(test_list_sorted, size//2), number=1000)
    
    # Format output
    o1_str = f"{time_o1:.6f}"
    on_str = f"{time_on:.6f}"
    on2_str = f"{time_on2:.6f}" if time_on2 != float('inf') else "TOO SLOW"
    ologn_str = f"{time_ologn:.6f}"
    
    print(f"{size}\t\t{o1_str}\t{on_str}\t{on2_str}\t{ologn_str}")

print("\nKey insights:")
print("- O(1): Performance stays constant regardless of input size")
print("- O(log n): Performance grows very slowly with input size")
print("- O(n): Performance grows linearly with input size")
print("- O(n²): Performance grows quadratically - becomes unusable quickly")

In [ ]:
# Optimizing data structures
print("\n=== OPTIMIZING DATA STRUCTURES ===")

import collections

# List vs Set vs Dict lookup performance
print("Lookup performance comparison:")

sizes = [1000, 10000, 100000]

for size in sizes:
    # Create test data
    data_list = list(range(size))
    data_set = set(range(size))
    data_dict = {i: i for i in range(size)}
    
    # Test lookups (look for the last element)
    target = size - 1
    
    # List lookup - O(n)
    time_list = timeit.timeit(lambda: target in data_list, number=100)
    
    # Set lookup - O(1) average
    time_set = timeit.timeit(lambda: target in data_set, number=100)
    
    # Dict lookup - O(1) average
    time_dict = timeit.timeit(lambda: target in data_dict, number=100)
    
    print(f"Size {size}:")
    print(f"  List lookup: {time_list:.6f}s")
    print(f"  Set lookup:  {time_set:.6f}s")
    print(f"  Dict lookup: {time_dict:.6f}s")
    print(f"  List is {time_list/time_set:.1f}x slower than set")
    print(f"  List is {time_list/time_dict:.1f}x slower than dict\n")

# Counter vs manual counting
print("Counter vs manual counting:")

def manual_count(arr):
    """Manual counting with dictionary."""
    counts = {}
    for item in arr:
        counts[item] = counts.get(item, 0) + 1
    return counts

def counter_count(arr):
    """Counting with collections.Counter."""
    return collections.Counter(arr)

# Test data
test_data = [1, 2, 3, 1, 2, 3, 1, 2, 3] * 1000

time_manual = timeit.timeit(lambda: manual_count(test_data), number=100)
time_counter = timeit.timeit(lambda: counter_count(test_data), number=100)

print(f"Manual counting: {time_manual:.6f}s")
print(f"Counter: {time_counter:.6f}s")
print(f"Counter is {time_manual/time_counter:.1f}x faster")

# Verify they produce the same result
manual_result = manual_count(test_data)
counter_result = counter_count(test_data)
print(f"Results match: {manual_result == dict(counter_result)}")

# Deque vs List for queue operations
print("\nDeque vs List for queue operations:")

def test_list_queue(n):
    """Test list as queue."""
    queue = []
    for i in range(n):
        queue.append(i)  # O(1)
    for _ in range(n):
        queue.pop(0)  # O(n) - inefficient!

def test_deque_queue(n):
    """Test deque as queue."""
    queue = collections.deque()
    for i in range(n):
        queue.append(i)  # O(1)
    for _ in range(n):
        queue.popleft()  # O(1) - efficient!

n = 10000
time_list = timeit.timeit(lambda: test_list_queue(n), number=10)
time_deque = timeit.timeit(lambda: test_deque_queue(n), number=10)

print(f"List as queue: {time_list:.6f}s")
print(f"Deque as queue: {time_deque:.6f}s")
print(f"Deque is {time_list/time_deque:.1f}x faster for queue operations")

## 💾 Memory Optimization

Efficient memory usage is crucial for high-performance applications.

In [ ]:
# Memory optimization techniques
print("=== MEMORY OPTIMIZATION TECHNIQUES ===")

import sys
import gc

# Check memory usage of different data structures
print("Memory usage comparison:")

# Different ways to store the same data
data_list = [i for i in range(10000)]
data_tuple = tuple(i for i in range(10000))
data_range = range(10000)

print(f"List memory: {sys.getsizeof(data_list)} bytes")
print(f"Tuple memory: {sys.getsizeof(data_tuple)} bytes")
print(f"Range memory: {sys.getsizeof(data_range)} bytes")

# Generator vs list comprehension
print("\nGenerator vs list comprehension:")

# List comprehension - stores all values in memory
list_comp = [x**2 for x in range(100000)]
print(f"List comprehension memory: {sys.getsizeof(list_comp)} bytes")

# Generator - lazy evaluation
generator = (x**2 for x in range(100000))
print(f"Generator memory: {sys.getsizeof(generator)} bytes")

# Calculate sum using both
sum_list = sum(list_comp)
sum_gen = sum(generator)
print(f"Results match: {sum_list == sum_gen}")

# __slots__ for memory-efficient classes
print("\n__slots__ for memory efficiency:")

class RegularClass:
    """Regular class with __dict__."""
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedClass:
    """Class with __slots__ for memory efficiency."""
    __slots__ = ['x', 'y']
    
    def __init__(self, x, y):
        self.x = x
        self.y = y

# Compare memory usage
regular_objects = [RegularClass(i, i*2) for i in range(10000)]
slotted_objects = [SlottedClass(i, i*2) for i in range(10000)]

# Calculate memory usage
regular_memory = sum(sys.getsizeof(obj) + sys.getsizeof(obj.__dict__) for obj in regular_objects)
slotted_memory = sum(sys.getsizeof(obj) for obj in slotted_objects)

print(f"Regular class memory: {regular_memory} bytes")
print(f"Slotted class memory: {slotted_memory} bytes")
print(f"Memory savings: {(regular_memory - slotted_memory) / regular_memory * 100:.1f}%")

# Test functionality
regular_obj = regular_objects[0]
slotted_obj = slotted_objects[0]

print(f"Regular object: x={regular_obj.x}, y={regular_obj.y}")
print(f"Slotted object: x={slotted_obj.x}, y={slotted_obj.y}")

# Dynamic attributes (only works with regular class)
try:
    regular_obj.z = 100  # This works
    print("✓ Regular class allows dynamic attributes")
except:
    print("✗ Regular class doesn't allow dynamic attributes")

try:
    slotted_obj.z = 100  # This fails
    print("✓ Slotted class allows dynamic attributes")
except AttributeError:
    print("✗ Slotted class doesn't allow dynamic attributes (by design)")

In [ ]:
# Memory leak detection
print("\n=== MEMORY LEAK DETECTION ===")

import weakref

# Example of a potential memory leak
class MemoryLeakExample:
    """Example class that might cause memory leaks."""
    
    instances = []
    
    def __init__(self, name):
        self.name = name
        # This creates a reference cycle!
        self.instances.append(self)
    
    @classmethod
    def clear_instances(cls):
        """Clear all instances."""
        cls.instances.clear()

# Create some objects
print("Creating objects with potential memory leak:")
objects = [MemoryLeakExample(f"Object {i}") for i in range(10)]
print(f"Created {len(objects)} objects")
print(f"Instances list has {len(MemoryLeakExample.instances)} objects")

# Delete references
del objects
gc.collect()  # Force garbage collection

# Check if objects are still in memory
print(f"After deletion, instances list still has {len(MemoryLeakExample.instances)} objects")
print("This is a memory leak! Objects can't be garbage collected.")

# Fix the leak
MemoryLeakExample.clear_instances()
gc.collect()
print(f"After clearing instances: {len(MemoryLeakExample.instances)} objects")

# Better implementation using weak references
class BetterExample:
    """Better implementation using weak references."""
    
    instances = weakref.WeakSet()  # WeakSet doesn't prevent garbage collection
    
    def __init__(self, name):
        self.name = name
        self.instances.add(self)

# Test the better implementation
print("\nTesting better implementation with weak references:")
better_objects = [BetterExample(f"Better Object {i}") for i in range(10)]
print(f"Created {len(better_objects)} better objects")
print(f"WeakSet has {len(BetterExample.instances)} objects")

# Delete references
del better_objects
gc.collect()

print(f"After deletion, WeakSet has {len(BetterExample.instances)} objects")
print("No memory leak! Objects were properly garbage collected.")

# Memory monitoring
print("\nMemory monitoring:")
print(f"Garbage collector stats: {gc.get_stats()}")
print(f"Uncollectable garbage: {len(gc.garbage)} items")

## 🚀 Caching and Memoization

Caching expensive operations can dramatically improve performance.

In [ ]:
# Memoization with functools.lru_cache
print("=== MEMOIZATION WITH LRU_CACHE ===")

import functools

# Expensive function without caching
def fibonacci_no_cache(n):
    """Fibonacci without caching - very slow for large n."""
    if n <= 1:
        return n
    return fibonacci_no_cache(n-1) + fibonacci_no_cache(n-2)

# Same function with caching
@functools.lru_cache(maxsize=None)
def fibonacci_cached(n):
    """Fibonacci with LRU cache - much faster."""
    if n <= 1:
        return n
    return fibonacci_cached(n-1) + fibonacci_cached(n-2)

# Test performance difference
test_values = [20, 25, 30, 35]

print("Performance comparison (Fibonacci):")
print("n\tWithout Cache\tWith Cache\tSpeedup")
print("-" * 50)

for n in test_values:
    # Time without cache
    if n <= 30:  # Don't wait too long for uncached version
        time_no_cache = timeit.timeit(lambda: fibonacci_no_cache(n), number=1)
    else:
        time_no_cache = float('inf')
    
    # Time with cache
    time_with_cache = timeit.timeit(lambda: fibonacci_cached(n), number=10)
    
    # Calculate speedup
    if time_no_cache != float('inf'):
        speedup = time_no_cache / (time_with_cache / 10)  # Adjust for number of runs
        speedup_str = f"{speedup:.0f}x"
        no_cache_str = f"{time_no_cache:.4f}s"
    else:
        speedup_str = "∞"
        no_cache_str = "TOO SLOW"
    
    print(f"{n}\t{no_cache_str}\t\t{time_with_cache:.6f}s\t{speedup_str}")

# Cache statistics
print(f"\nCache info: {fibonacci_cached.cache_info()}")
print(f"Cache size: {len(fibonacci_cached.cache)}")

# Verify correctness
for n in range(10):
    no_cache = fibonacci_no_cache(n)
    with_cache = fibonacci_cached(n)
    print(f"fib({n}) = {with_cache} (correct: {no_cache == with_cache})")

In [ ]:
# Custom caching implementations
print("\n=== CUSTOM CACHING IMPLEMENTATIONS ===")

# Simple dictionary cache
class SimpleCache:
    """Simple dictionary-based cache."""
    
    def __init__(self):
        self.cache = {}
    
    def get(self, key):
        return self.cache.get(key)
    
    def set(self, key, value):
        self.cache[key] = value
        return value

# LRU Cache implementation
class LRUCache:
    """Simple LRU cache implementation."""
    
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = {}
        self.order = []  # Track access order
    
    def get(self, key):
        if key in self.cache:
            # Move to end (most recently used)
            self.order.remove(key)
            self.order.append(key)
            return self.cache[key]
        return None
    
    def set(self, key, value):
        if key in self.cache:
            # Update existing
            self.order.remove(key)
        elif len(self.cache) >= self.capacity:
            # Evict least recently used
            lru_key = self.order.pop(0)
            del self.cache[lru_key]
        
        self.cache[key] = value
        self.order.append(key)
        return value

# Function caching decorator
def cached(cache_instance):
    """Decorator that uses a cache instance."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Create cache key
            key = (func.__name__, args, frozenset(kwargs.items()) if kwargs else None)
            
            # Check cache
            result = cache_instance.get(key)
            if result is not None:
                return result
            
            # Compute and cache
            result = func(*args, **kwargs)
            cache_instance.set(key, result)
            return result
        return wrapper
    return decorator

# Test custom caching
simple_cache = SimpleCache()
lru_cache = LRUCache(3)

# Expensive function
def expensive_computation(x, y):
    """Simulate expensive computation."""
    import time
    time.sleep(0.1)  # Simulate work
    return x ** y

# Create cached versions
@cached(simple_cache)
def cached_simple(x, y):
    return expensive_computation(x, y)

@cached(lru_cache)
def cached_lru(x, y):
    return expensive_computation(x, y)

# Test performance
print("Testing custom cache performance:")

# First calls (cache miss)
print("First calls (cache miss):")
start = time.time()
result1 = cached_simple(2, 10)
time1 = time.time() - start

start = time.time()
result2 = cached_lru(2, 10)
time2 = time.time() - start

print(f"Simple cache: {time1:.3f}s (result: {result1})")
print(f"LRU cache: {time2:.3f}s (result: {result2})")

# Second calls (cache hit)
print("\nSecond calls (cache hit):")
start = time.time()
result3 = cached_simple(2, 10)
time3 = time.time() - start

start = time.time()
result4 = cached_lru(2, 10)
time4 = time.time() - start

print(f"Simple cache: {time3:.6f}s (result: {result3})")
print(f"LRU cache: {time4:.6f}s (result: {result4})")
print(f"Cache speedup: {time1/time3:.0f}x faster on cache hit!")

# Test LRU eviction
print("\nTesting LRU eviction:")
for i in range(5):
    result = cached_lru(2, i)
    print(f"cached_lru(2, {i}) = {result}")

# Check if first item was evicted
result = cached_lru(2, 0)  # Should be a cache miss (recomputed)
print(f"cached_lru(2, 0) = {result} (should be slow if evicted)")

## 🔄 Multiprocessing for CPU-Bound Tasks

For CPU-bound tasks, multiprocessing can provide significant performance improvements.

In [ ]:
# Multiprocessing basics
print("=== MULTIPROCESSING BASICS ===")

import multiprocessing as mp
import os

# CPU-bound function
def cpu_intensive_task(n):
    """A CPU-intensive task for demonstration."""
    result = 0
    for i in range(n):
        result += i ** 3  # More intensive than i**2
    return result

# Single process execution
print(f"Running on {mp.cpu_count()} CPU cores")

print("\nSingle process execution:")
start_time = time.time()
results_single = [cpu_intensive_task(50000) for _ in range(4)]
single_time = time.time() - start_time
print(f"Single process time: {single_time:.2f}s")
print(f"Results: {results_single}")

# Multiprocess execution
print("\nMultiprocess execution:")
start_time = time.time()

# Create a pool of processes
with mp.Pool(processes=4) as pool:
    results_multi = pool.map(cpu_intensive_task, [50000] * 4)

multi_time = time.time() - start_time
print(f"Multiprocess time: {multi_time:.2f}s")
print(f"Results: {results_multi}")
print(f"Speedup: {single_time/multi_time:.2f}x")
print(f"Results match: {results_single == results_multi}")

# Process communication
print("\n=== PROCESS COMMUNICATION ===")

def worker_process(queue, lock, worker_id):
    """Worker process that communicates via queue."""
    with lock:
        print(f"Worker {worker_id} starting")
    
    # Do some work
    result = cpu_intensive_task(20000)
    
    # Send result back
    queue.put((worker_id, result))
    
    with lock:
        print(f"Worker {worker_id} finished")

# Test inter-process communication
queue = mp.Queue()
lock = mp.Lock()

processes = []
num_processes = 3

# Start worker processes
for i in range(num_processes):
    p = mp.Process(target=worker_process, args=(queue, lock, i))
    processes.append(p)
    p.start()

# Collect results
results = []
for _ in range(num_processes):
    worker_id, result = queue.get()
    results.append((worker_id, result))

# Wait for all processes to complete
for p in processes:
    p.join()

print(f"Collected results: {sorted(results)}")
print("All processes completed successfully")

## 🏁 Summary and Performance Best Practices

Congratulations! You've mastered Python performance optimization. Here's what you learned:

In [ ]:
# Performance Optimization Best Practices
print("=" * 60)
print("PYTHON PERFORMANCE OPTIMIZATION MASTERY - COMPLETE GUIDE")
print("=" * 60)

performance_practices = {
    "Profiling First": [
        "Always profile before optimizing - don't guess bottlenecks",
        "Use cProfile for CPU profiling",
        "Use memory_profiler for memory usage analysis",
        "Use timeit for accurate timing measurements",
        "Profile in production-like conditions"
    ],
    "Algorithm Optimization": [
        "Understand Big O complexity of algorithms",
        "Choose optimal data structures (dict vs list vs set)",
        "Use collections.deque for queue operations",
        "Prefer built-in functions over manual implementations",
        "Use generators for memory-efficient iteration"
    ],
    "Memory Optimization": [
        "Use __slots__ to reduce memory usage in classes",
        "Choose tuple over list when immutability is acceptable",
        "Use generators instead of list comprehensions for large data",
        "Avoid reference cycles that prevent garbage collection",
        "Use weak references when appropriate"
    ],
    "Caching Strategies": [
        "Use functools.lru_cache for function memoization",
        "Implement custom caching for complex scenarios",
        "Consider cache size limits and eviction policies",
        "Cache expensive database queries and API calls",
        "Invalidate cache when underlying data changes"
    ],
    "Multiprocessing": [
        "Use multiprocessing for CPU-bound tasks",
        "Async programming is better for I/O-bound tasks",
        "Be careful with shared state between processes",
        "Use multiprocessing.Pool for simple parallel tasks",
        "Consider process startup overhead"
    ],
    "Code Optimization": [
        "Avoid global variable lookups in loops",
        "Use local variables for frequently accessed attributes",
        "Minimize function call overhead in tight loops",
        "Use list comprehensions instead of map/filter",
        "Consider using C extensions for performance-critical code"
    ]
}

for category, practices in performance_practices.items():
    print(f"\n🚀 {category}:")
    for practice in practices:
        print(f"   ✓ {practice}")

print("\n" + "=" * 60)
print("🎯 PERFORMANCE OPTIMIZATION PHILOSOPHY")
print("=" * 60)
print("✓ Premature optimization is the root of all evil - measure first!")
print("✓ 80/20 rule: 80% of performance gains come from 20% of optimizations")
print("✓ Optimize algorithms before micro-optimizations")
print("✓ Balance performance with code readability and maintainability")
print("✓ Profile in the target environment (production-like conditions)")
print("✓ Consider the full system, not just individual functions")
print("✓ Document performance-critical code and optimization decisions")
print("=" * 60)

## 📚 Additional Resources

- **Python Profiling**: https://docs.python.org/3/library/profile.html
- **Memory Management**: https://docs.python.org/3/library/gc.html
- **Multiprocessing**: https://docs.python.org/3/library/multiprocessing.html
- **functools**: https://docs.python.org/3/library/functools.html
- **Performance Best Practices**: https://wiki.python.org/moin/PythonSpeed

## 💡 Pro Tips

1. **Profile First**: Always measure before optimizing
2. **Big O Matters**: Algorithm choice often matters more than micro-optimizations
3. **Data Structures**: Choose the right data structure for your access patterns
4. **Cache Wisely**: Cache expensive operations, but consider cache invalidation
5. **Memory Matters**: Memory usage affects performance, especially with large datasets
6. **Multiprocessing**: Use for CPU-bound tasks, async for I/O-bound tasks
7. **Built-ins**: Python built-ins are highly optimized - use them!
8. **Measure Everything**: Profile, benchmark, and measure memory usage
9. **Balance Trade-offs**: Performance vs. readability vs. maintainability
10. **Iterate**: Optimization is an iterative process

---

**Congratulations!** 🎉 You've mastered Python performance optimization! You now know how to profile, analyze, and optimize Python applications for maximum performance. Remember: measure first, then optimize based on data, not assumptions!